In [65]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import numpy as np

In [7]:
# tree-based models
x_train = pd.read_csv('../data/processed/x_train.csv')
x_cv = pd.read_csv('../data/processed/x_cv.csv')
x_test = pd.read_csv('../data/processed/x_test.csv')

y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_cv = pd.read_csv('../data/processed/y_cv.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# logistic regression and neural networks
x_train_scaled = pd.read_csv('../data/processed/x_train_scaled.csv')
x_cv_scaled = pd.read_csv('../data/processed/x_cv_scaled.csv')
x_test_scaled = pd.read_csv('../data/processed/x_test_scaled.csv')

cs_test = pd.read_csv('../data/processed/cs_test_processed.csv')
cs_test_scaled = pd.read_csv('../data/processed/cs_test_scaled.csv')

# for GridSearchCV and RandomizedSearchCV use x_train + x_cv
x_train_full = np.concatenate((x_train, x_cv), axis=0)
x_train_sc_full = np.concatenate((x_train_scaled, x_cv_scaled), axis=0)
y_train_full = np.concatenate((y_train, y_cv), axis=0)

In [27]:
scores = []

In [39]:
def build_search_cv(param_grid, model, x_train, y_train):
    if isinstance(model, (XGBClassifier, RandomForestClassifier)):
        grid = RandomizedSearchCV(model, param_grid, scoring=['roc_auc', 'f1'], refit='roc_auc', n_jobs=6)
    else:
        grid = GridSearchCV(model, param_grid, scoring=['roc_auc', 'f1'], refit='roc_auc')

    grid.fit(x_train, y_train)
    print(grid.best_params_, grid.best_score_)
    best_model = grid.best_estimator_
    scores.append([model, grid.best_score_])

    return best_model

In [ ]:
lr_params = {'solver': ['liblinear', 'lbfgs', 'sag']}
lr_model = build_search_cv(lr_params, LogisticRegression(random_state=42), x_train_sc_full, y_train_full)

/Users/nana/current proj/credit/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nana/current proj/credit/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nana/current proj/credit/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nana/current proj/credit/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


{'solver': 'sag'} 0.8390514783990739


/Users/nana/current proj/credit/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
dt_params = {'max_depth': [3, 5, 10, None], 'min_samples_split': [2, 7, 10, 15]}
dt_model = build_search_cv(dt_params, DecisionTreeClassifier(random_state=42), x_train_full, y_train_full)

{'max_depth': 5, 'min_samples_split': 2} 0.8474094770266062


In [45]:
rf_params = {'n_estimators': [100, 200, 400], 'max_depth': [5, 10, 12, 20], 'min_samples_split':[2, 5, 10, 15]}
rf_model = build_search_cv(rf_params, RandomForestClassifier(random_state=42, n_jobs=1), x_train_full, y_train_full)

{'n_estimators': 200, 'min_samples_split': 2, 'max_depth': 10} 0.8631926848671096


In [44]:
xgbc_params = {'n_estimators': [100, 200, 400, 700], 'learning_rate': [0.01, 0.03, 0.05], 'max_depth': [5, 10, 12, 20]}
xgbc_model = build_search_cv(xgbc_params, XGBClassifier(random_state=42, n_jobs=1), x_train_full, y_train_full)

{'n_estimators': 700, 'max_depth': 5, 'learning_rate': 0.01} 0.8663715356608215


In [52]:
best_pair = max(scores, key=lambda x: x[1])
best_pair

[XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=True, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=None,
               n_jobs=1, num_parallel_tree=None, ...),
 np.float64(0.8663715356608215)]

In [63]:
def build_models():
    model_1 = Sequential([
        Dense(units=25, activation='relu'),
        Dense(units=15, activation='relu'),
        Dense(units=1, activation='linear'),
    ], name='model_1')

    model_2 = Sequential([
        Dense(units=30, activation='relu'),
        Dense(units=12, activation='relu'),
        Dense(units=7, activation='relu'),
        Dense(units=1, activation='linear'),
    ], name='model_2')

    model_3 = Sequential([
        Dense(units=32, activation='relu'),
        BatchNormalization(),
        Dense(units=16, activation='relu'),
        BatchNormalization(),
        Dense(units=8, activation='relu'),
        BatchNormalization(),
        Dense(units=12, activation='relu'),
        BatchNormalization(),
        Dense(units=1, activation='linear'),
    ], name='model_3')

    models = [model_1, model_2, model_3]
    return models

In [70]:
models_nn = build_models()

early_stop = EarlyStopping(
    monitor='val_AUC',      
    patience=20,            
    restore_best_weights=True,  
    mode='max'               
)

results = {}
histories = {}

for model in models_nn:
    model.compile(
        loss=BinaryCrossentropy(from_logits=True),
        optimizer=Adam(learning_rate=0.01),
        metrics=['AUC'] 
    )

    model.fit(
        x_train_scaled, y_train,      
        validation_data=(x_cv_scaled, y_cv),
        epochs=200,             
        batch_size=256,
        callbacks=[early_stop],
        verbose=0
    )

    logits_train = model.predict(x_train_scaled, verbose=True).flatten()
    y_pred_train = tf.sigmoid(logits_train).numpy()

    logits_cv = model.predict(x_cv_scaled, verbose=True).flatten()
    y_pred_cv = tf.sigmoid(logits_cv).numpy()

    train_auc = roc_auc_score(y_train, y_pred_train)
    cv_auc = roc_auc_score(y_cv, y_pred_cv)
    print(f'Train AUC: {train_auc:.3f}, CV AUC: {cv_auc:.3f}, gap: {train_auc - cv_auc:.3f}')   

2802/2802 ━━━━━━━━━━━━━━━━━━━━ 1s 375us/step
934/934 ━━━━━━━━━━━━━━━━━━━━ 0s 493us/step
Train AUC: 0.872, CV AUC: 0.856, gap: 0.016
2802/2802 ━━━━━━━━━━━━━━━━━━━━ 1s 341us/step
934/934 ━━━━━━━━━━━━━━━━━━━━ 0s 314us/step
Train AUC: 0.869, CV AUC: 0.861, gap: 0.008
2802/2802 ━━━━━━━━━━━━━━━━━━━━ 2s 525us/step
934/934 ━━━━━━━━━━━━━━━━━━━━ 0s 386us/step
Train AUC: 0.865, CV AUC: 0.860, gap: 0.005
